In [13]:
import json
from pathlib import Path
from typing import Dict, Any, List, Iterable, Tuple


# ============================================================
# Config
# ============================================================
ROOT_DIR = Path("/root/paddlejob/workspace/env_run/output/yulinhao/projects/KnowRL/eval/data/Filtered")
CSS_DIR = ROOT_DIR / "css"
CBRS_DIR = ROOT_DIR / "cbrs"

INPUT_FILENAME = "test.jsonl"
OUTPUT_DIR = Path("/root/paddlejob/workspace/env_run/output/yulinhao/projects/KnowRL/eval/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "OpenMath_Nemotron_1_5B"
CSS_POLICY_NAME = "reverse_combination"
CBRS_POLICY_NAME = "oracle"
from datasets import Dataset

def write_parquet(path: Path, data: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    dataset = Dataset.from_list(data)
    dataset.to_parquet(str(path))

# ============================================================
# IO utils
# ============================================================
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line_id, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"[Warning] Failed to parse {path} line {line_id}: {e}")
    return data


def write_jsonl(path: Path, data: Iterable[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


# ============================================================
# Dataset discovery
# ============================================================
def get_dataset_dirs(base_dir: Path) -> Dict[str, Path]:
    dataset_map = {}
    if not base_dir.exists():
        return dataset_map

    for p in base_dir.iterdir():
        if p.is_dir():
            dataset_map[p.name] = p
    return dataset_map


def find_paired_datasets(
    css_dir: Path,
    cbrs_dir: Path,
) -> List[Tuple[str, Path, Path]]:
    css_datasets = get_dataset_dirs(css_dir)
    cbrs_datasets = get_dataset_dirs(cbrs_dir)

    common_names = sorted(set(css_datasets.keys()) & set(cbrs_datasets.keys()))
    pairs = [(name, css_datasets[name], cbrs_datasets[name]) for name in common_names]

    css_only = sorted(set(css_datasets.keys()) - set(cbrs_datasets.keys()))
    cbrs_only = sorted(set(cbrs_datasets.keys()) - set(css_datasets.keys()))

    if css_only:
        print(f"[Info] Only in css: {css_only}")
    if cbrs_only:
        print(f"[Info] Only in cbrs: {cbrs_only}")

    return pairs


# ============================================================
# Key / field extraction
# ============================================================

def normalize_prompt(item: Dict[str, Any]) -> List[Dict[str, str]]:
    """
    统一把样本里的 prompt/problem 转成:
    [
        {
            "content": "...",
            "role": "user"
        }
    ]
    """
    prompt = item.get("prompt")

    # 情况1：已经是你想要的 list[dict] 结构
    if isinstance(prompt, list) and len(prompt) > 0:
        first = prompt[0]
        if isinstance(first, dict) and "content" in first:
            content = str(first.get("content", "")).strip()
            role = str(first.get("role", "user")).strip() or "user"
            return [{"content": content, "role": role}]

    # 情况2：prompt 是一个 dict
    if isinstance(prompt, dict):
        content = str(prompt.get("content", "")).strip()
        role = str(prompt.get("role", "user")).strip() or "user"
        return [{"content": content, "role": role}]

    # 情况3：prompt 是纯字符串
    if isinstance(prompt, str):
        return [{"content": prompt.strip(), "role": "user"}]

    # 情况4：没有 prompt，但有 problem
    problem = item.get("problem")
    if isinstance(problem, str):
        return [{"content": problem.strip(), "role": "user"}]

    # 情况5：兜底
    raise ValueError(f"Unexpected prompt type: {type(prompt)} -- {prompt[:80]}")

def normalize_answer(item: Dict[str, Any]) -> Dict[str, str]:
    """
    将不同格式的答案统一为:
    {
        "ground_truth": str,
        "style": str
    }

    兼容情况：
    1. 已经有 reward_model（优先用）
    2. HMMT: answer 是数字 / 字符串
    3. 兜底：空字符串
    """

    # -------------------------------
    # Case 1: 已经有 reward_model
    # -------------------------------
    reward_model = item.get("reward_model")
    if isinstance(reward_model, dict):
        return {
            "ground_truth": str(reward_model.get("ground_truth", "") or ""),
            "style": str(reward_model.get("style", "") or ""),
        }

    # -------------------------------
    # Case 2: HMMT / 其他: answer 字段
    # -------------------------------
    answer = item.get("answer")

    if answer is not None:
        return {
            "ground_truth": str(answer),
            "style": "rule",   # HMMT没有style，留空
        }

    # -------------------------------
    # Case 3: 兜底
    # -------------------------------
    raise ValueError("Error")

def make_item_key(item: Dict[str, Any]) -> str:
    """
    用统一后的 prompt[0]["content"] 作为对齐键
    """
    prompt = normalize_prompt(item)
    return prompt[0]["content"]


def get_kept_kp_index(
    item: Dict[str, Any],
    model_name: str,
    policy_name: str,
) -> List[int]:
    """
    读取:
    item["kp_policy"][model_name][policy_name]["kept_kp_index"]
    """
    try:
        kept = item["kp_policy"][model_name][policy_name]["kept_kp_index"]
    except KeyError as e:
        raise KeyError(
            f"Missing kp_policy path: kp_policy -> {model_name} -> {policy_name} -> kept_kp_index"
        ) from e

    if kept is None:
        return []

    return sorted(kept)


def get_selected_kps_from_index(kp_list: List[Any], kept_kp_index: List[int]) -> List[Any]:
    """
    根据 kept_kp_index 从 kp_list 中取出对应知识点
    """
    selected = []
    for idx in kept_kp_index:
        if 0 <= idx < len(kp_list):
            selected.append(kp_list[idx])
    return selected


# ============================================================
# Core processing
# ============================================================
def process_dataset_pair(
    dataset_name: str,
    css_data: List[Dict[str, Any]],
    cbrs_data: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    css_map = {}
    for item in css_data:
        key = make_item_key(item)
        if key in css_map:
            print(f"[Warning] Duplicate key in css ({dataset_name}): {key[:80]}...")
        css_map[key] = item

    cbrs_map = {}
    for item in cbrs_data:
        key = make_item_key(item)
        if key in cbrs_map:
            print(f"[Warning] Duplicate key in cbrs ({dataset_name}): {key[:80]}...")
        cbrs_map[key] = item

    common_keys = sorted(set(css_map.keys()) & set(cbrs_map.keys()))
    css_only_keys = sorted(set(css_map.keys()) - set(cbrs_map.keys()))
    cbrs_only_keys = sorted(set(cbrs_map.keys()) - set(css_map.keys()))

    print(
        f"[{dataset_name}] "
        f"css={len(css_map)}, cbrs={len(cbrs_map)}, "
        f"common={len(common_keys)}, "
        f"css_only={len(css_only_keys)}, cbrs_only={len(cbrs_only_keys)}"
    )

    output_data = []

    for key in common_keys:
        css_item = css_map[key]
        cbrs_item = cbrs_map[key]

        # 默认认为同一道题的 kp_list 一致；以 css 的为准
        kp_list = css_item.get("extracted_knowledge_list")
        reward_model = normalize_answer(css_item)
        assert reward_model["ground_truth"]

        # 分别读取两个不同 policy 下的 kept_kp_index
        css_kept_kp_index = get_kept_kp_index(
            css_item,
            model_name=MODEL_NAME,
            policy_name=CSS_POLICY_NAME,
        )
        cbrs_kept_kp_index = get_kept_kp_index(
            cbrs_item,
            model_name=MODEL_NAME,
            policy_name=CBRS_POLICY_NAME,
        )

        # 可选：顺手转成选中的知识点
        css_selected_kps = get_selected_kps_from_index(kp_list, css_kept_kp_index)
        cbrs_selected_kps = get_selected_kps_from_index(kp_list, cbrs_kept_kp_index)

        merged_item = {
            "dataset": dataset_name,
            "prompt": normalize_prompt(css_item),
            "reward_model": reward_model,
            "kp_list": kp_list,

            "css_kept_kp_index": css_kept_kp_index,
            "css_selected_kps": css_selected_kps,

            "cbrs_kept_kp_index": cbrs_kept_kp_index,
            "cbrs_selected_kps": cbrs_selected_kps,
        }

        output_data.append(merged_item)
    # ========================================================
    # 统计平均知识点数量（per dataset）
    # ========================================================
    css_total = 0
    cbrs_total = 0
    count = 0

    for item in output_data:
        css_total += len(item["css_kept_kp_index"])
        cbrs_total += len(item["cbrs_kept_kp_index"])
        count += 1

    if count > 0:
        css_avg = css_total / count
        cbrs_avg = cbrs_total / count

        print(f"[{dataset_name}] Avg KP count:")
        print(f"  CSS  : {css_avg:.4f}")
        print(f"  CBRS : {cbrs_avg:.4f}")
        print(f"  Diff (CBRS - CSS): {cbrs_avg - css_avg:.4f}")
    else:
        print(f"[{dataset_name}] No valid samples")

    return output_data


# ============================================================
# Main
# ============================================================
def main():
    dataset_pairs = find_paired_datasets(CSS_DIR, CBRS_DIR)

    if not dataset_pairs:
        print("No paired datasets found.")
        return

    total_css = 0
    total_cbrs = 0
    total_count = 0

    for dataset_name, css_dataset_dir, cbrs_dataset_dir in dataset_pairs:
        css_file = css_dataset_dir / INPUT_FILENAME
        cbrs_file = cbrs_dataset_dir / INPUT_FILENAME

        if not css_file.exists():
            print(f"[Skip] Missing css file: {css_file}")
            continue
        if not cbrs_file.exists():
            print(f"[Skip] Missing cbrs file: {cbrs_file}")
            continue

        print(f"\nProcessing dataset: {dataset_name}")
        print(f"  CSS : {css_file}")
        print(f"  CBRS: {cbrs_file}")

        css_data = load_jsonl(css_file)
        cbrs_data = load_jsonl(cbrs_file)

        output_data = process_dataset_pair(
            dataset_name=dataset_name,
            css_data=css_data,
            cbrs_data=cbrs_data,
        )
        for item in output_data:
            total_css += len(item["css_kept_kp_index"])
            total_cbrs += len(item["cbrs_kept_kp_index"])
            total_count += 1

        output_dir = OUTPUT_DIR / dataset_name

        jsonl_path = output_dir / f"{dataset_name}.jsonl"
        parquet_path = output_dir / f"{dataset_name}.parquet"

        write_jsonl(jsonl_path, output_data)
        write_parquet(parquet_path, output_data)

        print(f"[Done] {dataset_name}")
        print(f"  jsonl   -> {jsonl_path}")
        print(f"  parquet -> {parquet_path}")

        
    print("\n===== Overall Statistics =====")

    if total_count > 0:
        css_avg = total_css / total_count
        cbrs_avg = total_cbrs / total_count

        print(f"Total samples: {total_count}")
        print(f"CSS  Avg KP: {css_avg:.4f}")
        print(f"CBRS Avg KP: {cbrs_avg:.4f}")
        print(f"Diff (CBRS - CSS): {cbrs_avg - css_avg:.4f}")
    else:
        print(f"No valid samples found. (total_count: {total_count})")


if __name__ == "__main__":
    main()

AttributeError: 'str' object has no attribute 'mkdir'

In [9]:
import json
from pathlib import Path
from datasets import Dataset

file_path = Path("/root/paddlejob/workspace/env_run/output/yulinhao/projects/KnowRL/eval/data/Olympiad_Bench/Olympiad_Bench.jsonl")

# 读取 jsonl
data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

# 转 Dataset
dataset = Dataset.from_list(data)

# 输出 parquet 路径（同目录）
output_path = file_path.with_suffix(".parquet")

# 保存
dataset.to_parquet(str(output_path))

print(f"Saved to: {output_path}")

Creating parquet from Arrow format:   0%|                                                                               | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format: 100%|███████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 86.17ba/s]

Saved to: /root/paddlejob/workspace/env_run/output/yulinhao/projects/KnowRL/eval/data/Olympiad_Bench/Olympiad_Bench.parquet
